# Silver: Parse and clean hourly data
This notebook reads the latest valid Bronze records for the requested date range, unpacks the raw JSON strings into hourly rows, normalizes timestamps to `UTC`, and merges the cleaned data into `silver.electricity_price_hourly` and `silver.weather_hourly`.

### Key design choices in this notebook
* **Separate source cleaning from joining:** Cleaning each source into its own Silver table before joining in `04_silver_join` makes debugging easier and keeps the cleaned weather and electricity tables reusable on their own.
* **Latest valid Bronze version (`latest_successful`):** If a date was fetched multiple times and the upstream data changed, a window function (`latest_window`) picks only the most recent valid (`HTTP 200`, `record_count > 0`) payload per `request_key`.
* **UTC timezone normalization:** The electricity API sends local Norwegian time (`+01:00` or `+02:00`), while MET Frost sends `UTC` (`Z`). Converting both to `time_start_utc` in a `UTC` Spark session avoids daylight saving time issues when joining. Also keeps `time_start_local` (`Europe/Oslo`) for human readability.
* **Weather sensor deduplication and pivoting:** MET Frost returns data in long format and can have multiple sensor readings per hour. We rank readings by `quality_code` and `time_series_id` to pick one value per element per hour, then pivot `air_temperature`, `wind_speed`, and `sum(precipitation_amount PT1H)` into columns.
* **Two-tier data quality (`ERROR` vs. `WARNING`):** Missing primary keys or physically impossible values (negative wind speed or precipitation) log an `ERROR` and stop the run, while extreme but possible weather values (`< -40°C`, `> 40°C`, or wind `> 60 m/s`) log a `WARNING` without stopping the pipeline.

In [0]:
import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType, StructType, StructField, StringType,
    DoubleType, IntegerType, BooleanType, TimestampType
)
from pyspark.sql.window import Window

# Lock Spark session to UTC and dynamically resolve the active Unity Catalog
spark.conf.set("spark.sql.session.timeZone", "UTC")
CATALOG = spark.sql("SELECT current_catalog()").first()[0]

# Read runtime parameters from job/notebook widgets (generates a UUID if run_id is empty)
dbutils.widgets.text("date_from", "2025-01-01")
dbutils.widgets.text("date_to", "2025-01-07")
dbutils.widgets.text("run_id", "")

DATE_FROM = dbutils.widgets.get("date_from")
DATE_TO = dbutils.widgets.get("date_to")
RUN_ID = dbutils.widgets.get("run_id") or str(uuid.uuid4())

### Helper functions for Bronze version ranking, uniqueness checks, and Delta upserts
* **`latest_successful`:** Filters Bronze to `HTTP 200` and `record_count > 0` in the date range, and uses `row_number()` over `request_key` ordered by `retrieved_at_utc DESC` to pick only the newest valid payload per day.
* **`assert_unique`:** Fails the run immediately if duplicate business keys exist before merging.
* **`upsert_all`:** Runs an idempotent Delta `MERGE` (`whenMatchedUpdateAll` + `whenNotMatchedInsertAll`) on the Silver business keys.
* **`write_dq`:** Logs `ERROR` and `WARNING` check results to `ops.data_quality_results`.

In [0]:
# Rank Bronze records per request key so the most recently retrieved payload comes first
latest_window = Window.partitionBy("request_key").orderBy(
    F.col("retrieved_at_utc").desc(),
    F.col("bronze_record_id").desc()
)

# Fetch only the latest valid (HTTP 200, non-empty) Bronze record for each request key in the date range
def latest_successful(table_name):
    return (
        spark.table(table_name)
        .filter(
            (F.col("request_date").between(DATE_FROM, DATE_TO))
            & (F.col("http_status") == 200)
            & (F.col("record_count") > 0)
        )
        .withColumn("version_rank", F.row_number().over(latest_window))
        .filter("version_rank = 1")
        .drop("version_rank")
    )

# Fail the pipeline if the DataFrame contains duplicate rows for the specified business keys
def assert_unique(df, key_columns, label):
    duplicate_count = (
        df.groupBy(*key_columns).count().filter("count > 1").count()
    )
    if duplicate_count:
        raise ValueError(
            f"{label} contains {duplicate_count} duplicate business keys"
        )

# Perform an idempotent Delta MERGE that updates existing rows and inserts new ones
def upsert_all(source_df, table_name, merge_condition):
    (
        DeltaTable.forName(spark, table_name).alias("target")
        .merge(source_df.alias("source"), merge_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# Log a single data quality check result into the centralized ops.data_quality_results table
def write_dq(check_name, severity, passed, actual_value, message):
    row = [(
        RUN_ID, "silver_transform", check_name, severity, passed,
        float(actual_value) if actual_value is not None else None,
        message, datetime.now(timezone.utc).replace(tzinfo=None)
    )]
    schema = """
        run_id string, layer string, check_name string, severity string,
        passed boolean, actual_value double, message string,
        checked_at_utc timestamp
    """
    df = spark.createDataFrame(row, schema)
    (
        DeltaTable.forName(
            spark, f"{CATALOG}.ops.data_quality_results"
        ).alias("target")
        .merge(
            df.alias("source"),
            """
            target.run_id = source.run_id
            AND target.layer = source.layer
            AND target.check_name = source.check_name
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

### Parse and clean electricity prices (`silver.electricity_price_hourly`)
Parses the daily JSON array using `price_schema`, explodes the array into hourly rows, converts `time_start` (with `+01:00`/`+02:00` offset) to `time_start_utc`, checks for duplicates and nulls, and merges into `silver.electricity_price_hourly` on `(price_area, time_start_utc)`.

In [0]:
# JSON schema for parsing the raw Hvakosterstrommen array string stored in Bronze
price_schema = ArrayType(StructType([
    StructField("NOK_per_kWh", DoubleType(), True),
    StructField("EUR_per_kWh", DoubleType(), True),
    StructField("EXR", DoubleType(), True),
    StructField("time_start", StringType(), True),
    StructField("time_end", StringType(), True)
]))

# Parse JSON, explode daily arrays into hourly rows, and convert local timestamps to UTC
price_df = (
    latest_successful(f"{CATALOG}.bronze.electricity_price_api_raw")
    .withColumn("items", F.from_json("payload_json", price_schema))
    .select(
        "bronze_record_id", "retrieved_at_utc", "price_area",
        F.explode("items").alias("item")
    )
    .select(
        "price_area",
        F.to_timestamp(
            "item.time_start", "yyyy-MM-dd'T'HH:mm:ssXXX"
        ).alias("time_start_utc"),
        F.to_timestamp(
            "item.time_end", "yyyy-MM-dd'T'HH:mm:ssXXX"
        ).alias("time_end_utc"),
        F.col("item.NOK_per_kWh").cast("double").alias("nok_per_kwh"),
        F.col("item.EUR_per_kWh").cast("double").alias("eur_per_kwh"),
        F.col("item.EXR").cast("double").alias("exchange_rate"),
        F.col("bronze_record_id").alias("source_bronze_record_id"),
        F.col("retrieved_at_utc").alias("source_retrieved_at_utc")
    )
    .withColumn(
        "time_start_local",
        F.from_utc_timestamp("time_start_utc", "Europe/Oslo")
    )
    .withColumn("processed_at_utc", F.current_timestamp())
)

# Fail fast if there are duplicate (price_area, time_start_utc) keys or NULL required fields
assert_unique(price_df, ["price_area", "time_start_utc"], "Price Silver")

price_required_errors = price_df.filter(
    F.col("price_area").isNull()
    | F.col("time_start_utc").isNull()
    | F.col("nok_per_kwh").isNull()
).count()

write_dq(
    "price_required_fields", "ERROR", price_required_errors == 0,
    price_required_errors, "Price area, timestamp, and NOK price are required"
)

if price_required_errors:
    raise ValueError("Price Silver contains null required fields")

# Upsert cleaned hourly prices into silver.electricity_price_hourly
upsert_all(
    price_df,
    f"{CATALOG}.silver.electricity_price_hourly",
    """
    target.price_area = source.price_area
    AND target.time_start_utc = source.time_start_utc
    """
)

### Parse, deduplicate, and pivot weather observations (`silver.weather_hourly`)
1. **Unpack nested JSON (`weather_long`):** Parses the nested `data -> observations` arrays from MET Frost into one row per observation (`long` format).
2. **Deduplicate sensor readings (`selected`):** Uses `selection_window` per `(station_id, time_start_utc, element_id)` to pick the reading with the best `quality_code` (lowest number) and primary `time_series_id`.
3. **Pivot to wide format (`values` & `quality`):** Pivots the three weather elements into hourly columns (`air_temperature_c`, `wind_speed_mps`, `precipitation_1h_mm`) along with their quality codes and Bronze lineage IDs.
4. **Validate and merge:** Stops on missing keys or negative wind/precipitation (`ERROR`), logs extreme weather values (`WARNING`), and merges into `silver.weather_hourly` on `(station_id, time_start_utc)`.

In [0]:
# JSON schema for unpacking nested MET Frost observations (data[] -> observations[])
weather_schema = StructType([
    StructField("data", ArrayType(StructType([
        StructField("sourceId", StringType(), True),
        StructField("referenceTime", StringType(), True),
        StructField("observations", ArrayType(StructType([
            StructField("elementId", StringType(), True),
            StructField("value", DoubleType(), True),
            StructField("unit", StringType(), True),
            StructField("qualityCode", IntegerType(), True),
            StructField("timeOffset", StringType(), True),
            StructField("timeResolution", StringType(), True),
            StructField("timeSeriesId", IntegerType(), True)
        ])), True)
    ])), True)
])

# Flatten the nested JSON arrays into long format (one row per weather element per hour)
weather_long = (
    latest_successful(f"{CATALOG}.bronze.weather_api_raw")
    .withColumn("parsed", F.from_json("payload_json", weather_schema))
    .select(
        "bronze_record_id", "retrieved_at_utc", "station_id",
        F.explode_outer("parsed.data").alias("data")
    )
    .select(
        "bronze_record_id", "retrieved_at_utc", "station_id",
        F.to_timestamp("data.referenceTime").alias("time_start_utc"),
        F.explode_outer("data.observations").alias("obs")
    )
    .select(
        "bronze_record_id", "retrieved_at_utc", "station_id",
        "time_start_utc",
        F.col("obs.elementId").alias("element_id"),
        F.col("obs.value").cast("double").alias("value"),
        F.col("obs.qualityCode").alias("quality_code"),
        F.col("obs.timeSeriesId").alias("time_series_id"),
        F.col("obs.timeResolution").alias("time_resolution"),
        F.col("obs.timeOffset").alias("time_offset")
    )
    .filter(F.col("time_start_utc").isNotNull())
)

# If multiple readings exist for the same hour and element, pick the one with the best qualityCode
selection_window = Window.partitionBy(
    "station_id", "time_start_utc", "element_id"
).orderBy(
    F.col("quality_code").asc_nulls_last(),
    F.col("time_series_id").asc_nulls_last(),
    F.col("bronze_record_id").desc()
)

selected = (
    weather_long
    .withColumn("selection_rank", F.row_number().over(selection_window))
    .filter("selection_rank = 1")
    .drop("selection_rank")
)

# Pivot the three weather elements from rows (long) into hourly columns (wide)
values = (
    selected.groupBy("station_id", "time_start_utc")
    .pivot("element_id", [
        "air_temperature",
        "wind_speed",
        "sum(precipitation_amount PT1H)"
    ])
    .agg(F.first("value"))
    .withColumnRenamed("air_temperature", "air_temperature_c")
    .withColumnRenamed("wind_speed", "wind_speed_mps")
    .withColumnRenamed(
        "sum(precipitation_amount PT1H)", "precipitation_1h_mm"
    )
)

# Also pivot the qualityCode for each element so downstream users can see sensor quality
quality = (
    selected.groupBy("station_id", "time_start_utc")
    .pivot("element_id", [
        "air_temperature",
        "wind_speed",
        "sum(precipitation_amount PT1H)"
    ])
    .agg(F.first("quality_code"))
    .withColumnRenamed("air_temperature", "temperature_quality_code")
    .withColumnRenamed("wind_speed", "wind_quality_code")
    .withColumnRenamed(
        "sum(precipitation_amount PT1H)", "precipitation_quality_code"
    )
)

# Keep Bronze lineage IDs (source_bronze_record_id and source_retrieved_at_utc) per hour
source_lineage = (
    selected.groupBy("station_id", "time_start_utc")
    .agg(
        F.max("bronze_record_id").alias("source_bronze_record_id"),
        F.max("retrieved_at_utc").alias("source_retrieved_at_utc")
    )
)

# Combine pivoted values, quality codes, and lineage into one wide row per UTC hour
weather_df = (
    values.join(quality, ["station_id", "time_start_utc"], "left")
    .join(source_lineage, ["station_id", "time_start_utc"], "left")
    .withColumn(
        "time_start_local",
        F.from_utc_timestamp("time_start_utc", "Europe/Oslo")
    )
    .withColumn("processed_at_utc", F.current_timestamp())
    .select(
        "station_id", "time_start_utc", "time_start_local",
        "air_temperature_c", "wind_speed_mps", "precipitation_1h_mm",
        "temperature_quality_code", "wind_quality_code",
        "precipitation_quality_code", "source_bronze_record_id",
        "source_retrieved_at_utc", "processed_at_utc"
    )
)

# Check key uniqueness and fail fast on NULL keys or physically impossible negative wind/precipitation
assert_unique(weather_df, ["station_id", "time_start_utc"], "Weather Silver")

weather_key_errors = weather_df.filter(
    F.col("station_id").isNull() | F.col("time_start_utc").isNull()
).count()

negative_weather = weather_df.filter(
    (F.col("wind_speed_mps") < 0)
    | (F.col("precipitation_1h_mm") < 0)
).count()

write_dq(
    "weather_keys_present", "ERROR", weather_key_errors == 0,
    weather_key_errors, "Weather station and UTC timestamp are required"
)
write_dq(
    "nonnegative_wind_and_precipitation", "ERROR",
    negative_weather == 0, negative_weather,
    "Observed wind speed and precipitation must not be negative"
)

if weather_key_errors or negative_weather:
    raise ValueError("Weather Silver failed required data-quality checks")

# Log a non-blocking WARNING if extreme weather values occur (retained without failing the run)
unusual_weather = weather_df.filter(
    (F.col("air_temperature_c") < -40)
    | (F.col("air_temperature_c") > 40)
    | (F.col("wind_speed_mps") > 60)
).count()

write_dq(
    "unusual_weather_values", "WARNING", unusual_weather == 0,
    unusual_weather,
    "Unusual values are retained but should be reviewed"
)

# Upsert cleaned hourly weather observations into silver.weather_hourly
upsert_all(
    weather_df,
    f"{CATALOG}.silver.weather_hourly",
    """
    target.station_id = source.station_id
    AND target.time_start_utc = source.time_start_utc
    """
)

print(f"Silver source transformation completed. run_id={RUN_ID}")
display(price_df.orderBy("time_start_utc").limit(5))
display(weather_df.orderBy("time_start_utc").limit(5))